In [1]:
# importamos librerías
# para manipulación de datos
import pandas as pd
import numpy as np
# para trabajar con APIs
import requests
# para gestionar archivos y directorios
import os
import zipfile
# para manipular pdfs
import camelot
# para trabajar con fechas
from datetime import datetime

In [ ]:
def descargar_datos_cmadrid(url, carpeta_salida, formato = None, mes = False, dia = False):
    """
    Descarga archivos desde una URL y los guarda en la carpeta especificada.

    Args:
        url (str): URL de la API que proporciona los datos a descargar.
        carpeta_salida (str): Ruta de la carpeta donde se guardarán los archivos.
        formato (str, opcional): Formato de archivo deseado (ej. "csv"). Si es None, descarga cualquier formato disponible.
        mes (bool, opcional): Si es True, incluye el mes en el nombre del archivo.
        dia (bool, opcional): Si es True, incluye el día en el nombre del archivo.

    Returns:
        list: Lista con los nombres de los archivos descargados exitosamente.

    Raises:
        requests.exceptions.RequestException: Si hay un problema con la solicitud HTTP.
        Exception: Para otros errores inesperados en el proceso de descarga.

    """

    hoy =  datetime.now().date()
    response = requests.get(url)
    archivos_descargados = []
    if response.status_code == 200:
        data = response.json()
        info = data.get("result", {}).get("resources")
        archivos_guardados = 0
        archivos_totales = len(info)
        formato_usuario = formato.lower() if formato else None
        for elemento in info:
            formato_archivo = elemento.get("format").lower()
            nombre = elemento.get("name")
            if formato_usuario is None or formato_archivo == formato_usuario:
                datos = elemento.get("url")
                response = requests.get(datos)
                # Verificar si la descarga fue exitosa
                if response.status_code == 200:
                    if not mes and not dia:
                        nombre = f'{carpeta_salida}/cmadrid_{nombre}.{formato_archivo}'
                    elif mes and not dia:
                        nombre = f'{carpeta_salida}/cmadrid_{hoy.month}_{hoy.year}.{formato_archivo}'
                    elif dia and not mes:
                        nombre = f'{carpeta_salida}/cmadrid_{hoy}.{formato_archivo}'
                    # Guardar el archivo en tu computadora
                    with open(nombre, 'wb') as file:
                        file.write(response.content) 
                    archivos_guardados +=1 
                    archivos_descargados.append(elemento["name"])   
            else:
                    print(f"No se descargó el archivo {elemento["name"]}.{elemento["format"]}. Código de estado: {response.status_code}")
    print(f"{archivos_guardados} de {archivos_totales} archivos guardados exitosamente.")
    return archivos_descargados

In [67]:
historicos_cmadrid = descargar_datos_cmadrid("https://datos.comunidad.madrid/api/3/action/package_show?id=calidad_aire_datos_historico")
historicos_madrid = historicos_cmadrid[1:]

22 de 22 archivos guardados exitosamente.


In [16]:
def descargar_datos_madrid(url, carpeta_salida):
    """
    Descarga archivos de calidad del aire desde la API del Ayuntamiento de Madrid.
    
    Parámetros:
    - url (str): URL de la API donde se encuentran los datos de calidad del aire.
    - carpeta_salida (str): Ruta de la carpeta donde se guardarán los archivos descargados.

    La función consulta la API, filtra los archivos relevantes y los guarda en la carpeta indicada
    """
    # Obtenemos la fecha actual
    hoy =  datetime.now().date()  
    # hacemos perición a la API
    response = requests.get(url)
    # si la respuesta es exitosa obtenemos los datos en formato JSON
    if response.status_code == 200:
        data = response.json()
        elementos = data.get("result", {}).get("items") # accedemos a la lista de archivos que necesitamos
        # iteramos por cada elemento en la lista para filtrar los que necesitamos por su título
        for elemento in elementos:
            # Caso 1: "Calidad del aire. Estaciones de control"
            if elemento["title"] == "Calidad del aire. Estaciones de control":
                datos_est = elemento.get("distribution")
                for i in datos_est:
                    print(i.get("title"))
                    formato = i.get("format",{}).get("value").split('/')[-1]
                    if formato.lower() == "csv": # solo descargamos el csv
                        archivo = i.get("accessURL")
                        print(f"Descargando: {archivo} como madrid_{elemento["title"]}.{formato}")
                        response_est = requests.get(archivo)
                        if response_est.status_code == 200:
                            with open(f'{carpeta_salida}/madrid_{elemento["title"]}.{formato}', 'wb') as file:
                                file.write(response_est.content)
                                print("Archivo guardado exitosamente.")
                        else:
                            print(f"No se pudo descargar el archivo {elemento["title"]}. Código de estado: {response.status_code}")
             # Caso 2: "Calidad del aire. Datos en tiempo real acumulado"               
            elif elemento["title"] == "Calidad del aire. Datos en tiempo real acumulado":
                datos_tra = elemento.get("distribution")
                for i in datos_tra:
                    print(i.get("title"))
                    formato = i.get("format",{}).get("value").split('/')[-1]
                    if "csv" in i.get("title"): #descargamos solo el que tiene csv en su título
                        archivo = i.get("accessURL")
                        print(f"Descargando: {archivo} como madrid_{elemento["title"]}.{formato}")
                        response_tra = requests.get(archivo)
                        if response_tra.status_code == 200:
                            with open(f'{carpeta_salida}/madrid_{hoy}.{formato}', 'wb') as file:
                                file.write(response_tra.content)
                                print("Archivo guardado exitosamente.")
                        else:
                            print(f"No se pudo descargar el archivo {elemento["title"]}. Código de estado: {response.status_code}")
            # Caso 3: "Calidad del aire. Datos horarios desde 2001"
            elif elemento["title"] == "Calidad del aire. Datos horarios desde 2001":
                archivos_dh = []
                datos = elemento.get("distribution")
                for i in datos:
                    nombre_archivo = i.get("title")
                    formato = i.get("format",{}).get("value").split('/')[-1]
                    archivo = i.get("accessURL")
                    # Verificar si el título es solo un año
                    if nombre_archivo.isdigit() and len(nombre_archivo) == 4:  # Asegura que es un año (ej. "2024")
                         descargar = True
                         nombre = nombre_archivo
                         archivos_dh.append(nombre)
                    else:
                        descargar = formato.lower() == "csv"  # Si tiene más texto, solo descarga CSV
                        nombre = nombre_archivo.split(" ")[1]
                    if descargar:
                        print(f"Descargando: {archivo} como madrid_{nombre}.{formato}")
                        response_dh = requests.get(archivo)
                        if response_dh.status_code == 200:
                             with open(f'{carpeta_salida}/madrid_{nombre}.{formato}', 'wb') as file:
                                file.write(response_dh.content)
                                
                                print("Archivo guardado exitosamente.")
                        else:
                            print(f"No se pudo descargar el archivo {nombre}. Código de estado: {response_dh.status_code}")

    return archivos_dh

In [17]:
archivos_madrid = descargar_datos_madrid("https://datos.madrid.es/egob/catalogo/keyword/aire.json", "../data/raw")

Calidad del aire. Tiempo real xml
Calidad del aire. Tiempo real acumulado json
Calidad del aire. Tiempo real csv
Descargando: https://datos.madrid.es/egob/catalogo/300755-12751583-calidad-aire-tiempo-real-acumula.csv como madrid_Calidad del aire. Datos en tiempo real acumulado.csv
Archivo guardado exitosamente.
Consulta el API de datos.madrid.es
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306322-calidad-aire-horario.csv como madrid_2025.csv
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306320-calidad-aire-horario.zip como madrid_2024.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306319-calidad-aire-horario.zip como madrid_2023.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306318-calidad-aire-horario.zip como madrid_2022.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306317-calidad-a

In [18]:
archivos_madrid

['2024',
 '2023',
 '2022',
 '2021',
 '2020',
 '2019',
 '2018',
 '2017',
 '2016',
 '2015',
 '2014',
 '2013',
 '2012',
 '2011',
 '2010',
 '2009',
 '2008',
 '2007',
 '2006',
 '2005',
 '2004',
 '2003',
 '2002',
 '2001']

In [19]:
def procesar_archivos_zip(archivos_dh, carpeta_salida):
    """
    Procesa archivos ZIP que contienen datos en formato CSV, extrae los archivos,
    los concatena y guarda el resultado en un nuevo CSV.

    Args:
        archivos_dh (list): Lista con los nombres de los archivos ZIP a procesar.
        carpeta_salida (str): Ruta de la carpeta donde se guardarán los archivos procesados.

    Returns:
        None: Guarda los archivos CSV concatenados en la carpeta indicada.
    """
    # Crear la carpeta de salida si no existe
    os.makedirs(carpeta_salida, exist_ok=True)

    # Diccionarios para almacenar listas de DataFrames y nombres de CSV
    listas_dfs = {}
    listas_csv = {}

    for archivo in archivos_dh:
        ruta_zip = os.path.join(carpeta_salida, f"madrid_{archivo}.zip")  # Construir la ruta al archivo ZIP
        listas_dfs[archivo] = []  # Inicializar una lista de DataFrames para este archivo
        listas_csv[archivo] = []  # Inicializar una lista de nombres de CSV para este archivo

        # Abrir el ZIP y cargar los CSV
        with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
            listas_csv[archivo] = [nombre for nombre in zip_ref.namelist() if nombre.endswith(".csv")]

            for nombre_csv in listas_csv[archivo]:
                with zip_ref.open(nombre_csv) as f:
                    df = pd.read_csv(f, sep=";", index_col=0)
                    listas_dfs[archivo].append(df)
        # Concatenar los DataFrames de la lista en uno solo
        df_concatenado = pd.concat(listas_dfs[archivo],  ignore_index=True)

        # Guardar el DataFrame concatenado en un archivo CSV
        ruta_salida = os.path.join(carpeta_salida, f"madrid_{archivo}.csv")
        df_concatenado.to_csv(ruta_salida, sep=";", index=False)

        print(f"Archivo concatenado guardado en: {ruta_salida}")

        # Eliminar el archivo ZIP
        os.remove(ruta_zip)
        print(f"Archivo ZIP eliminado: {ruta_zip}")

In [20]:
madrid_procesado=procesar_archivos_zip(archivos_madrid, carpeta_salida="../data/raw")

Archivo concatenado guardado en: ../data/raw\madrid_2024.csv
Archivo ZIP eliminado: ../data/raw\madrid_2024.zip
Archivo concatenado guardado en: ../data/raw\madrid_2023.csv
Archivo ZIP eliminado: ../data/raw\madrid_2023.zip
Archivo concatenado guardado en: ../data/raw\madrid_2022.csv
Archivo ZIP eliminado: ../data/raw\madrid_2022.zip
Archivo concatenado guardado en: ../data/raw\madrid_2021.csv
Archivo ZIP eliminado: ../data/raw\madrid_2021.zip
Archivo concatenado guardado en: ../data/raw\madrid_2020.csv
Archivo ZIP eliminado: ../data/raw\madrid_2020.zip
Archivo concatenado guardado en: ../data/raw\madrid_2019.csv
Archivo ZIP eliminado: ../data/raw\madrid_2019.zip
Archivo concatenado guardado en: ../data/raw\madrid_2018.csv
Archivo ZIP eliminado: ../data/raw\madrid_2018.zip
Archivo concatenado guardado en: ../data/raw\madrid_2017.csv
Archivo ZIP eliminado: ../data/raw\madrid_2017.zip
Archivo concatenado guardado en: ../data/raw\madrid_2016.csv
Archivo ZIP eliminado: ../data/raw\madrid_2

In [ ]:
def extraer_tabla_pdf(archivo_pdf, pagina, nombre, carpeta_salida):
    """
    Extrae la primera tabla de una página específica de un archivo PDF y la guarda como un archivo CSV.

    Args:
        archivo_pdf (str): Ruta del archivo PDF del cual se extraerá la tabla.
        pagina (str): Número de la página que contiene la tabla.
        nombre (str): Nombre del archivo CSV resultante.
        carpeta_salida (str): Ruta donde se guardará el archivo CSV.

    Returns:
        bool: True si la extracción y el guardado de la tabla fueron exitosos, False si no se encontraron tablas en la página indicada.

    Raises:
        FileNotFoundError: Se lanza si el archivo PDF especificado no existe.
        Exception: Captura y maneja errores inesperados durante el proceso de extracción.

    """

    try:
        # Extraer tablas de la página indicada
        tablas = camelot.read_pdf(archivo_pdf, pages=pagina)

        # Verificar si se extrajeron tablas
        if len(tablas) > 0:
            tablas[0].to_csv(f"{carpeta_salida}/{nombre}.csv")  # Guardar la primera tabla como CSV
            print(f"✅ Tabla guardada en {carpeta_salida}")
            return True
        else:
            print("⚠️ No se encontraron tablas en la página indicada.")
            return False

    except FileNotFoundError:
        print(f"❌ Error: No se encontró el archivo {archivo_pdf}")
    except Exception as e:
        print(f"❌ Error inesperado: {e}")